# Fine-tuning check notebook

Quick checks that the evaluation and fine-tuning code behaves as expected. The real code lives in `backend/`, `evaluation/`, `scripts/` and `training/`; this notebook only calls it.

In [ ]:
import sys, csv
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
print(ROOT)

## 1. Phone numbers come through in every format

In [ ]:
from backend.text_numbers import extract_mobile_number
for text in ['Mon numéro est 0612345678', 'Mon numéro est 06 12 34 56 78', '+33 6 12 34 56 78',
             'zéro six douze trente-quatre cinquante-six soixante-dix-huit', 'My number is 9876543210']:
    print(f'{text!r:70} -> {extract_mobile_number(text, "fr")}')

## 2. Normalisation before WER

In [ ]:
from evaluation.text_normalizer import normalize, align
ref = "Euh, mon numéro c'est le 06 12 34 56 78."
hyp = 'mon numéro c est le zéro six douze trente quatre cinquante six soixante dix huit'
print(normalize(ref, 'fr'))
print(normalize(hyp, 'fr'))
a = align(normalize(ref, 'fr'), normalize(hyp, 'fr'))
print('WER', a.wer, 'S/D/I', a.substitutions, a.deletions, a.insertions)

## 3. Dataset splits

In [ ]:
for name in ['train', 'validation', 'test']:
    rows = list(csv.DictReader(open(ROOT / f'dataset/{name}.csv', encoding='utf-8')))
    langs = {}
    for r in rows:
        langs[r['language']] = langs.get(r['language'], 0) + 1
    print(f"{name:10} {len(rows):5} clips  {langs}  speakers={len({r['speaker_id'] for r in rows})}")

## 4. Phone channel

Compare the share of energy above 3.6 kHz before and after (should drop to ~0).

In [ ]:
import numpy as np
from scripts.augment_telephony import load_audio, telephone_channel
clip = next((ROOT / 'dataset/french/audio').glob('*.wav'))
audio = load_audio(clip)
phone = telephone_channel(audio, 16000, snr_db=15)
for label, x in [('clean', audio), ('phone', phone)]:
    power = np.abs(np.fft.rfft(x)) ** 2
    freqs = np.fft.rfftfreq(len(x), 1 / 16000)
    print(label, round(power[freqs > 3600].sum() / power.sum(), 4))

In [ ]:
# Listen to both (works inside Jupyter)
from IPython.display import Audio, display
display(Audio(audio, rate=16000)); display(Audio(phone, rate=16000))

## 5. Baseline on the test split

Needs `faster-whisper`. Results are appended to `experiments/results/summary.csv`.

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'evaluation.run_wer', '--metadata', 'dataset/test.csv',
                '--models', 'base', 'small'], cwd=ROOT, check=True)

In [ ]:
rows = list(csv.DictReader(open(ROOT / 'experiments/results/summary.csv', encoding='utf-8')))
for r in rows[-8:]:
    print(r['model'], r['language'], 'WER', r['wer'], 'phone', r['phone_accuracy'] or '-', 'tel', r['telephone'])